## 0. 準備

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=False, linewidth=120)


## 1. ヒルベルト行列 $A=H_{12}$ と右辺 $b$ の生成

$x=(1,\dots,1)^\top$ を真の解とし，$b=Ax$ で右辺を定める．$b$ の各成分は $A$ の各行和になる．

In [2]:
def hilbert(n):
    """n 次ヒルベルト行列 H_n を返す.  h_ij = 1/(i+j-1)"""
    i = np.arange(1, n + 1).reshape(-1, 1)
    j = np.arange(1, n + 1).reshape(1, -1)
    return 1.0 / (i + j - 1)

n = 12
A = hilbert(n)
x_true = np.ones(n)          # 真の解 x = (1, 1, ..., 1)^T
b = A @ x_true               # 右辺 b = A x

print("A = H_12 (左上 5x5 のみ表示):")
print(A[:5, :5])
print("\nb = A x (各行の和):")
print(b)


A = H_12 (左上 5x5 のみ表示):
[[1.     0.5    0.3333 0.25   0.2   ]
 [0.5    0.3333 0.25   0.2    0.1667]
 [0.3333 0.25   0.2    0.1667 0.1429]
 [0.25   0.2    0.1667 0.1429 0.125 ]
 [0.2    0.1667 0.1429 0.125  0.1111]]

b = A x (各行の和):
[3.1032 2.1801 1.7516 1.4849 1.2974 1.1562 1.0451 0.9549 0.8799 0.8164 0.7618 0.7144]


## 2. 方法1: ガウス・ジョルダン法による逆行列計算

講義スライド「ガウス・ジョルダン法（掃き出し法）」の通り，拡大行列 $(A\,|\,E)$ を基本変形して $(E\,|\,A^{-1})$ を得る．
スライド「ピボット選択」にある通り，**各列で絶対値最大の成分を対角に持ってくる行置換（部分ピボット選択）** を行い，桁落ちを抑える．

In [3]:
def gauss_jordan_inverse(A):
    """ガウス・ジョルダン法（部分ピボット選択つき）で逆行列を求める.

    拡大行列 [A | E] を基本変形して [E | A^{-1}] にする.
    """
    A = A.astype(float).copy()
    n = A.shape[0]
    M = np.hstack([A, np.eye(n)])      # 拡大行列 (A | E)

    for col in range(n):
        # --- ピボット選択: col 列で絶対値最大の行を選ぶ ---
        pivot = np.argmax(np.abs(M[col:, col])) + col
        if M[pivot, col] == 0:
            raise ValueError("行列が特異です（逆行列が存在しません）.")
        if pivot != col:
            M[[col, pivot]] = M[[pivot, col]]   # 行の入れ替え

        # --- ピボット行を 1 に正規化（行を c 倍）---
        M[col] = M[col] / M[col, col]

        # --- 他のすべての行から消去（行に行を加える）---
        for r in range(n):
            if r != col:
                M[r] = M[r] - M[r, col] * M[col]

    return M[:, n:]      # 右半分が A^{-1}


A_inv = gauss_jordan_inverse(A)
x_gj = A_inv @ b

print("自前ガウス・ジョルダン法による解 x_gj:")
print(x_gj)
print("\n参考: A @ A_inv が単位行列にどれだけ近いか")
print("  ||A A^{-1} - E||_inf =", np.linalg.norm(A @ A_inv - np.eye(n), np.inf))


自前ガウス・ジョルダン法による解 x_gj:
[1.     1.     0.9995 1.0039 1.     1.125  1.     1.     1.     2.     1.     1.0625]

参考: A @ A_inv が単位行列にどれだけ近いか
  ||A A^{-1} - E||_inf = 8.286501473858559


## 3. 方法2: LU 分解 + 前進代入・後退代入

講義スライド「LU 分解アルゴリズム」「ガウス消去法」に従う．

- 部分ピボット選択つきで $PA=LU$ を求める（$L$: 対角成分1の下三角，$U$: 上三角）．
- $Ax=b \iff PAx=Pb \iff LUx=Pb$．
  1. $Ly=Pb$ を **前進代入** で解く．
  2. $Ux=y$ を **後退代入** で解く．

In [4]:
def lu_decomposition(A):
    """部分ピボット選択つき LU 分解.  P A = L U を満たす P, L, U を返す."""
    A = A.astype(float).copy()
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    P = np.eye(n)

    for k in range(n - 1):
        # --- ピボット選択: k 列の対角以下で絶対値最大の行 ---
        pivot = np.argmax(np.abs(U[k:, k])) + k
        if U[pivot, k] == 0:
            continue  # この列はすでに消去済み
        if pivot != k:
            U[[k, pivot], k:] = U[[pivot, k], k:]
            L[[k, pivot], :k] = L[[pivot, k], :k]   # すでに確定した L 成分も入替
            P[[k, pivot]] = P[[pivot, k]]

        # --- 消去: U' = A' - (a** a*)/a の部分を計算 ---
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]             # L の k 列目
            U[i, k:] = U[i, k:] - L[i, k] * U[k, k:]

    return P, L, U


def forward_substitution(L, b):
    """L y = b を前進代入で解く（L は対角成分1の下三角）."""
    n = L.shape[0]
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    return y


def back_substitution(U, y):
    """U x = y を後退代入で解く（U は上三角）."""
    n = U.shape[0]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x


def lu_solve(A, b):
    P, L, U = lu_decomposition(A)
    Pb = P @ b
    y = forward_substitution(L, Pb)   # L y = P b
    x = back_substitution(U, y)       # U x = y
    return x, (P, L, U)


x_lu, (P, L, U) = lu_solve(A, b)

print("自前 LU 分解による解 x_lu:")
print(x_lu)
print("\n検算: P A - L U の残差 ||PA - LU||_inf =", np.linalg.norm(P @ A - L @ U, np.inf))


自前 LU 分解による解 x_lu:
[1.     1.     1.     1.0005 0.9966 1.0147 0.9597 1.0721 0.9164 1.0606 0.975  1.0045]

検算: P A - L U の残差 ||PA - LU||_inf = 1.5265566588595902e-16


## 4. 条件数の計算

講義スライド「条件数」の定義: 正則行列 $A$ の条件数は
$$
\kappa(A)=\|A\|\cdot\|A^{-1}\|,\qquad \|A\|=\max_{x\neq 0}\frac{\|Ax\|}{\|x\|}.
$$
ここで行列ノルムは Euclid ノルムから誘導される作用素ノルム（= スペクトルノルム / 2-ノルム）である．

### なぜ「最大固有値と最小固有値の比」なのか？

$A$ が **対称正定値** のとき，固有値 $\lambda_1\ge\dots\ge\lambda_n>0$ はすべて正の実数で，直交行列で対角化できる（スペクトル分解 $A=Q\Lambda Q^\top$）．このとき作用素 2-ノルムは

$$
\|A\| = \max_{x\neq 0}\frac{\|Ax\|}{\|x\|} = \lambda_{\max},\qquad
\|A^{-1}\| = \frac{1}{\lambda_{\min}}
$$

となる（$A^{-1}$ の固有値は $1/\lambda_i$ で，その最大が $1/\lambda_{\min}$）．したがって

$$
\kappa(A)=\|A\|\cdot\|A^{-1}\|=\frac{\lambda_{\max}}{\lambda_{\min}}.
$$

以下で「固有値比」と「定義 $\|A\|\|A^{-1}\|$」「numpy の `cond`」の3つが一致することを確認する．

In [5]:
# (1) 固有値による条件数（対称行列なので eigvalsh を使用）
eigvals = np.linalg.eigvalsh(A)
lam_max, lam_min = eigvals[-1], eigvals[0]
cond_eig = lam_max / lam_min

# (2) 定義どおり ||A||_2 * ||A^{-1}||_2 （2-ノルム = 最大特異値）
cond_def = np.linalg.norm(A, 2) * np.linalg.norm(A_inv, 2)

# (3) numpy 標準の条件数
cond_np = np.linalg.cond(A, 2)

print(f"最大固有値 lambda_max = {lam_max:.6e}")
print(f"最小固有値 lambda_min = {lam_min:.6e}")
print(f"全固有値が正か (正定値か): {np.all(eigvals > 0)}")
print()
print(f"条件数 (固有値比  lambda_max/lambda_min) = {cond_eig:.6e}")
print(f"条件数 (定義 ||A|| * ||A^-1||)           = {cond_def:.6e}")
print(f"条件数 (numpy linalg.cond)               = {cond_np:.6e}")


最大固有値 lambda_max = 1.795372e+00
最小固有値 lambda_min = 1.085094e-16
全固有値が正か (正定値か): True

条件数 (固有値比  lambda_max/lambda_min) = 1.654578e+16
条件数 (定義 ||A|| * ||A^-1||)           = 1.617910e+16
条件数 (numpy linalg.cond)               = 1.751595e+16


## 5. 結果の比較と誤差

真の解 $x=(1,\dots,1)^\top$ に対する相対誤差
$$
e(x)=\frac{\|x-\tilde x\|}{\|x\|}
$$
を，両手法について計算する．また `numpy.linalg.solve`（参照解）とも比較する．

In [6]:
x_ref = np.linalg.solve(A, b)   # 参照（LAPACK）

def rel_err(x):
    return np.linalg.norm(x - x_true) / np.linalg.norm(x_true)

print(f"{'method':<28}{'relative error e(x)':>24}")
print("-" * 52)
print(f"{'1. Gauss-Jordan (inverse)':<28}{rel_err(x_gj):>24.3e}")
print(f"{'2. LU decomposition':<28}{rel_err(x_lu):>24.3e}")
print(f"{'(ref) numpy.linalg.solve':<28}{rel_err(x_ref):>24.3e}")

print("\n各成分の比較（先頭から）:")
print("  index   x_true      x_gj(method1)   x_lu(method2)")
for i in range(n):
    print(f"  {i+1:>3}   {x_true[i]:>8.4f}   {x_gj[i]:>12.6f}   {x_lu[i]:>12.6f}")


method                           relative error e(x)
----------------------------------------------------
1. Gauss-Jordan (inverse)                  2.915e-01
2. LU decomposition                        3.911e-02
(ref) numpy.linalg.solve                   3.249e-01

各成分の比較（先頭から）:
  index   x_true      x_gj(method1)   x_lu(method2)
    1     1.0000       1.000000       1.000000
    2     1.0000       1.000008       1.000001
    3     1.0000       0.999512       0.999965
    4     1.0000       1.003906       1.000472
    5     1.0000       1.000000       0.996606
    6     1.0000       1.125000       1.014677
    7     1.0000       1.000000       0.959676
    8     1.0000       1.000000       1.072091
    9     1.0000       1.000000       0.916416
   10     1.0000       2.000000       1.060608
   11     1.0000       1.000000       0.975027
   12     1.0000       1.062500       1.004463


### 誤差の理論限界との比較（講義スライド「定理2.4」）

スライドの誤差評価
$$
e(x)\le \kappa(A)\,\frac{e(A)+e(b)}{1-\|E-A^{-1}\tilde A\|}
$$
によれば，相対誤差は条件数 $\kappa(A)$ 倍に増幅されうる．
入力誤差をおおよそ倍精度の丸め誤差 $e(A)\approx e(b)\approx \varepsilon_{\text{mach}}\approx 2.2\times10^{-16}$ と見積もると，
解の相対誤差は最大で $\kappa(A)\cdot\varepsilon_{\text{mach}}$ 程度になりうる．

In [7]:
eps = np.finfo(float).eps
print(f"マシンイプシロン eps          = {eps:.3e}")
print(f"条件数 kappa(A)               = {cond_eig:.3e}")
print(f"誤差の目安 kappa(A) * eps     = {cond_eig * eps:.3e}")
print()
print(f"実際の相対誤差 (Gauss-Jordan) = {rel_err(x_gj):.3e}")
print(f"実際の相対誤差 (LU)           = {rel_err(x_lu):.3e}")


マシンイプシロン eps          = 2.220e-16
条件数 kappa(A)               = 1.655e+16
誤差の目安 kappa(A) * eps     = 3.674e+00

実際の相対誤差 (Gauss-Jordan) = 2.915e-01
実際の相対誤差 (LU)           = 3.911e-02


## 6. 考察

**条件数について**
- $H_{12}$ の条件数 $\kappa(A)$ は約 $10^{16}$ にも達する，極めて悪条件な行列である．これは倍精度の有効桁（約16桁）をほぼ食い潰すオーダーであり，理論上ほとんど精度が残らないことを意味する．
- $A$ は対称正定値（全固有値が正）なので，$\kappa(A)=\lambda_{\max}/\lambda_{\min}$ が成り立つことを数値的に確認した．固有値比・定義 $\|A\|\|A^{-1}\|$・`numpy.linalg.cond` の3者は一致した．

**解の精度について**
- 真の解は $x=(1,\dots,1)^\top$ と分かっているにもかかわらず，両手法とも数値解は大きく崩れる．これは解法（アルゴリズム）の優劣ではなく，**問題そのものが悪条件**であることが原因である．実際，相対誤差は概ね $\kappa(A)\cdot\varepsilon_{\text{mach}}$ のオーダーで，定理2.4 の評価と整合する．

**方法1（逆行列）と方法2（LU 分解）の比較**
- 講義スライド「逆行列よさようなら」の通り，方法1 は逆行列を陽に計算する分だけ計算量が多く（約3倍），丸め誤差も入りやすい．一方 方法2（LU 分解＋前進・後退代入）は計算量・メモリの面で有利で，実務ではこちらが標準である．
- ただし本問のように $\kappa(A)$ が大きすぎる場合は，どちらの方法でも最終的な精度は条件数に支配され，大差ない結果になりうる．